In [2]:
from datetime import datetime,timedelta
from tldhamburg.backend.Things import Things,DBMaintainer,DataFetcher
import sqlite3
import requests
import time
import json
import os
from concurrent.futures import ThreadPoolExecutor

query = "SELECT thingID, primary_signal from DATASTREAMS"
with sqlite3.connect(DBMaintainer.get_db_path()) as connection:
    cursor = connection.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    cursor.execute(query)
    rows = cursor.fetchall()
    thing_ids = [ID[0] for ID in rows]
    datastream_ids = [ID[1] for ID in rows]


In [3]:
print(len(datastream_ids))

20021


In [4]:
# Function to fetch data
def fetch_data(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()  # Return parsed JSON
    except requests.RequestException as e:
        return {"error": str(e)}
    
combined_observations = dict()
start_idx = 20020 #last collected index 
counter = 0
for ID in datastream_ids[start_idx:]:
    start = time.time()
    urls = [
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=0&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=1000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=2000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=3000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=4000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=5000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=6000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=7000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=8000&$top=1000&$orderby=phenomenonTime asc",
        f"https://tld.iot.hamburg.de/v1.1/Datastreams({ID})/Observations?$skip=9000&$top=1000&$orderby=phenomenonTime asc",
    ]

    # Run requests in parallel
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(fetch_data, urls))

    # Combine results into a single list
    combined_results = list()
    for result in results:
        if isinstance(result, dict) and "value" in result:
            combined_results.extend(result["value"])

    observations = [(observation["phenomenonTime"],observation["result"]) for observation in combined_results]

    combined_observations[ID] = observations
    end = time.time()
    counter += 1
    print(f"count: {start_idx+counter}, ID: {ID}, fetch time: {end-start}")
    time.sleep(1)
    if counter%1 == 0:
        try:
            path = f"{os.getcwd()}"
            name = f"observations_ids{start_idx+counter-9}-{start_idx+counter}.json"
            file_path = os.path.join(path,name)
            with open(file_path,"w") as file:
                json.dump(combined_observations, file, indent=4)
            print(f"Data successfully saved to {file}")
            combined_observations = dict()
        except Exception as e:
            print(f"An error occurred while saving to file: {e}")

count: 20021, ID: 91099, fetch time: 0.21477508544921875
Data successfully saved to <_io.TextIOWrapper name='/Users/mk/Library/CloudStorage/OneDrive-Personal/Studium/WS24_25/Projekt/ci-s-projekt-2024-mikko/tldhamburg/backend/Notebooks/observations_ids20012-20021.json' mode='w' encoding='UTF-8'>


In [5]:
try:
    combined_data = dict()
    path = f"{os.getcwd()}"
    for i in range(2003):
        if i%100 ==0:
            print(i)
        name = f"observations_ids{10*i+1}-{10*(i+1)}.json"
        file_path = os.path.join(path,name)
        with open(file_path, "r") as input_file:
            data = json.load(input_file)
            combined_data.update(data)
    name = f"observations_ids20021.json"
    file_path = os.path.join(path,name)
    with open(file_path, "r") as input_file:
        data = json.load(input_file)
        combined_data.update(data)

    name = f"observations_all_datastreams.json"
    file_path = os.path.join(path,name)
    with open(file_path,"w") as combined_file:
        json.dump(combined_data,combined_file,indent=4)
except Exception as e:
    print(f"An error occurred while saving to file: {e}")

0
An error occurred while saving to file: [Errno 2] No such file or directory: '/Users/mk/Library/CloudStorage/OneDrive-Personal/Studium/WS24_25/Projekt/ci-s-projekt-2024-mikko/tldhamburg/backend/Notebooks/observations_ids1-10.json'


In [6]:
try:
    name = f"observations_ids20021.json"
    file_path = os.path.join(path,name)
    with open(file_path, "r") as input_file:
        data = json.load(input_file)
        combined_data.update(data)

    name = f"observations_all_datastreams.json"
    file_path = os.path.join(path,name)
    with open(file_path,"w") as combined_file:
        print("saving...")
        json.dump(combined_data,combined_file,indent=4)
        print("saved!")
except Exception as e:
    print(f"An error occurred while saving to file: {e}")

An error occurred while saving to file: [Errno 2] No such file or directory: '/Users/mk/Library/CloudStorage/OneDrive-Personal/Studium/WS24_25/Projekt/ci-s-projekt-2024-mikko/tldhamburg/backend/Notebooks/observations_ids20021.json'


In [7]:
sum_green = timedelta(0)
sum_red = timedelta(0)

last_green_observation_time = None
last_red_observation_time = None

green_percentages = dict()
for key,value in combined_data.items():
    for observation in value:
        if observation[1] == 1:
            last_red_observation_time = observation[0]
            if not last_green_observation_time:
                continue
            time_difference = datetime.strptime(last_red_observation_time,"%Y-%m-%dT%H:%M:%SZ") - datetime.strptime(last_green_observation_time,"%Y-%m-%dT%H:%M:%SZ")
            if time_difference < timedelta(minutes=5):
                sum_green += time_difference
        if observation[1] == 3 or observation[1] == 0:
            last_green_observation_time = observation[0]
            if not last_red_observation_time:
                continue
            time_difference = datetime.strptime(last_green_observation_time,"%Y-%m-%dT%H:%M:%SZ") - datetime.strptime(last_red_observation_time,"%Y-%m-%dT%H:%M:%SZ")
            if time_difference < timedelta(minutes=5):
                    sum_red += time_difference
    
    green_percentage = sum_green / (sum_green+sum_red)
    print(key)
    green_percentages[key] = green_percentage
